In [ ]:
!pip install tqdm opencv-python-headless pyyaml -q

In [ ]:
import os
import random
import shutil
import yaml
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

In [ ]:
TILE_WIDTH     = 608
TILE_POSITIONS = [0, 495, 991]
BBOX_PAD       = 3
MIN_AREA: list[int] = [175, 340, 1250, 1900]


def rle_decode(rle_string: str | None, shape: tuple[int, int] = (256, 1600)) -> np.ndarray:
    mask = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    if not rle_string:
        return mask.reshape(shape)
    s               = list(map(int, rle_string.split()))
    starts, lengths = s[0::2], s[1::2]
    for start, length in zip(starts, lengths):
        start -= 1
        mask[start:start + length] = 1
    return mask.reshape(shape, order='F')


def build_mask(rle_list: list[str | None], shape: tuple[int, int] = (256, 1600)) -> np.ndarray:
    if not rle_list:
        return np.zeros((*shape, 0), dtype=np.uint8)
    return np.stack([rle_decode(rle, shape) for rle in rle_list], axis=-1)


def tiling(image: np.ndarray, mask: np.ndarray) -> list[tuple[np.ndarray, np.ndarray]]:
    tiles = []
    for x_start in TILE_POSITIONS:
        x_end = min(x_start + TILE_WIDTH, image.shape[1])
        tiles.append((image[:, x_start:x_end], mask[:, x_start:x_end]))
    return tiles


def extract_boxes(mask_tile: np.ndarray) -> list[tuple[int, float, float, float, float]]:
    H, W, C = mask_tile.shape
    bboxes  = []
    for class_id in range(C):
        binary     = mask_tile[:, :, class_id].astype(np.uint8)
        num_labels, _, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
        for i in range(1, num_labels):
            x, y, w, h, area = stats[i]
            if area < MIN_AREA[class_id]:
                continue
            x1    = max(0, x - BBOX_PAD)
            y1    = max(0, y - BBOX_PAD)
            x2    = min(W, x + w + BBOX_PAD)
            y2    = min(H, y + h + BBOX_PAD)
            w_pad = x2 - x1
            h_pad = y2 - y1
            if w_pad <= 0 or h_pad <= 0:
                continue
            bboxes.append((
                class_id,
                (x1 + x2) / 2 / W,
                (y1 + y2) / 2 / H,
                w_pad / W,
                h_pad / H,
            ))
    return bboxes


def save_yolo_labels(bboxes: list[tuple[int, float, float, float, float]], file_path: str | Path) -> None:
    with open(file_path, 'w') as f:
        for cls, x, y, w, h in bboxes:
            f.write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

print('decoder loaded.')

In [ ]:
RAW_IMAGES_DIR = Path('/kaggle/input/competitions/severstal-steel-defect-detection/train_images')
CSV_PATH       = Path('/kaggle/input/competitions/severstal-steel-defect-detection/train.csv')
OUT_IMAGES_DIR = Path('/kaggle/working/processed_data/images')
OUT_LABELS_DIR = Path('/kaggle/working/processed_data/labels')
NUM_CLASSES    = 4


def load_annotations(csv_path: Path) -> dict[str, list[str | None]]:
    df = pd.read_csv(csv_path)
    annotations: dict[str, list] = {}
    for _, row in df.iterrows():
        img_id   = row['ImageId']
        class_id = int(row['ClassId']) - 1
        rle      = row['EncodedPixels']
        if img_id not in annotations:
            annotations[img_id] = [None] * NUM_CLASSES
        annotations[img_id][class_id] = rle if isinstance(rle, str) else None
    return annotations


def process_image(img_path: Path, rle_list: list[str | None]) -> dict:
    image = cv2.imread(str(img_path))
    if image is None:
        return {'tiles': 0, 'skipped': True}
    mask  = build_mask(rle_list)
    tiles = tiling(image, mask)
    stem  = img_path.stem
    stats = {'tiles': 0, 'defect_tiles': 0, 'empty_tiles': 0}
    for idx, (img_tile, mask_tile) in enumerate(tiles):
        tile_name = f'{stem}_tile{idx}'
        cv2.imwrite(str(OUT_IMAGES_DIR / f'{tile_name}.jpg'), img_tile)
        bboxes    = extract_boxes(mask_tile)
        save_yolo_labels(bboxes, OUT_LABELS_DIR / f'{tile_name}.txt')
        stats['tiles'] += 1
        if bboxes:
            stats['defect_tiles'] += 1
        else:
            stats['empty_tiles'] += 1
    return stats


OUT_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
OUT_LABELS_DIR.mkdir(parents=True, exist_ok=True)

print('Loading annotations...')
annotations = load_annotations(CSV_PATH)
all_images  = sorted(RAW_IMAGES_DIR.glob('*.jpg'))
print(f'Found images: {len(all_images)}')

total_tiles = defect_tiles = empty_tiles = skipped = 0

for img_path in tqdm(all_images, desc='Processing', unit='img'):
    img_id   = img_path.name
    rle_list = annotations.get(img_id, [None] * NUM_CLASSES)
    result   = process_image(img_path, rle_list)
    if result.get('skipped'):
        skipped += 1
        continue
    total_tiles  += result['tiles']
    defect_tiles += result['defect_tiles']
    empty_tiles  += result['empty_tiles']

print(f'\n=== Converter completed ===')
print(f'Images skipped:   {skipped}')
print(f'Tiles created:    {total_tiles}')
print(f'  with defects:   {defect_tiles}')
print(f'  defect-free:    {empty_tiles}')

In [ ]:
IMAGES_DIR     = Path('/kaggle/working/processed_data/images')
LABELS_DIR     = Path('/kaggle/working/processed_data/labels')
TRAIN_RATIO    = 0.75
VAL_RATIO      = 0.15
TEST_RATIO     = 0.10
SEED           = 42
CLASS_PRIORITY = [1, 0, 3, 2]
CLASS_NAMES    = ['crazing', 'inclusion', 'patches', 'pitted_surface']


def load_image_classes(csv_path: Path) -> dict[str, set[int]]:
    df = pd.read_csv(csv_path).dropna(subset=['EncodedPixels'])
    image_classes: dict[str, set[int]] = defaultdict(set)
    for _, row in df.iterrows():
        image_classes[row['ImageId']].add(int(row['ClassId']) - 1)
    return dict(image_classes)


def assign_stratum(classes: set[int]) -> int:
    for class_id in CLASS_PRIORITY:
        if class_id in classes:
            return class_id
    return -1


def split_images(all_stems: list[str], image_classes: dict[str, set[int]]) -> tuple[list, list, list]:
    strata: dict[int, list[str]] = defaultdict(list)
    for stem in all_stems:
        classes = image_classes.get(f'{stem}.jpg', set())
        strata[assign_stratum(classes)].append(stem)
    train, val, test = [], [], []
    for stems in strata.values():
        random.shuffle(stems)
        n       = len(stems)
        n_val   = max(1, round(n * VAL_RATIO))
        n_test  = max(1, round(n * TEST_RATIO))
        n_train = n - n_val - n_test
        train  += stems[:n_train]
        val    += stems[n_train:n_train + n_val]
        test   += stems[n_train + n_val:]
    return train, val, test


def move_tiles(stems: list[str], split: str) -> None:
    out_images = IMAGES_DIR / split
    out_labels = LABELS_DIR / split
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)
    for stem in tqdm(stems, desc=f'Moving {split}', unit='img'):
        for tile_idx in range(3):
            tile_name = f'{stem}_tile{tile_idx}'
            src_img   = IMAGES_DIR / f'{tile_name}.jpg'
            src_lbl   = LABELS_DIR / f'{tile_name}.txt'
            if src_img.exists():
                shutil.move(str(src_img), str(out_images / f'{tile_name}.jpg'))
            if src_lbl.exists():
                shutil.move(str(src_lbl), str(out_labels / f'{tile_name}.txt'))


random.seed(SEED)
image_classes = load_image_classes(CSV_PATH)
all_stems     = sorted({p.stem.rsplit('_tile', 1)[0] for p in IMAGES_DIR.glob('*.jpg')})
print(f'Found unique images: {len(all_stems)}')

train, val, test = split_images(all_stems, image_classes)
total = len(train) + len(val) + len(test)
print(f'  Train: {len(train):>5}  ({len(train)/total*100:.1f}%)')
print(f'  Val:   {len(val):>5}  ({len(val)/total*100:.1f}%)')
print(f'  Test:  {len(test):>5}  ({len(test)/total*100:.1f}%)')

move_tiles(train, 'train')
move_tiles(val,   'val')
move_tiles(test,  'test')

yaml_path = Path('/kaggle/working/data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump({
        'path' : '/kaggle/working/processed_data',
        'train': 'images/train',
        'val'  : 'images/val',
        'test' : 'images/test',
        'nc'   : len(CLASS_NAMES),
        'names': CLASS_NAMES,
    }, f, default_flow_style=False, sort_keys=False)

print(f'data.yaml saved to: {yaml_path}')
print('Done.')